# MQLoss Deep Dive: Pinball Loss, Calibration, and Distribution Choices

**Goal:** Understand how MQLoss works under the hood and develop intuition for choosing quantile levels and loss functions.

**Key Questions:**
- What does the pinball loss actually optimize?
- How do different `level=` settings affect the forecast uncertainty bands?
- How do you check whether your intervals are well-calibrated?
- When should you use DistributionLoss instead of MQLoss?

**Time:** ~13 minutes

---
**Prerequisites:** Guide 02 (Training Quantile Models), Notebook 01 (Quantiles Not Enough)

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss, DistributionLoss

np.random.seed(42)

# Load French Bakery data
url = "https://raw.githubusercontent.com/nicholasjmorales/French-Bakery-Daily-Transactional-Dataset/main/Bakery_sales.csv"
raw = pd.read_csv(url, parse_dates=["date"])

baguettes = (
    raw[raw["article"].str.upper().str.contains("BAGUETTE")]
    .groupby("date")["Quantity"]
    .sum()
    .reset_index()
    .rename(columns={"date": "ds", "Quantity": "y"})
    .assign(unique_id="bakery_total")
    .sort_values("ds")
    .reset_index(drop=True)
)

# Train/test split: last 28 days as test
cutoff = baguettes["ds"].max() - pd.Timedelta(days=28)
train_df = baguettes[baguettes["ds"] <= cutoff]
test_df = baguettes[baguettes["ds"] > cutoff]

print(f"Training samples: {len(train_df)} days")
print(f"Test samples:     {len(test_df)} days")
print(f"\nDemand stats (training):")
print(train_df["y"].describe().round(1))

## Part 1: Understanding the Pinball Loss

The pinball loss is an asymmetric loss function. The asymmetry is what makes it produce quantiles rather than means.

In [ ]:
def pinball_loss(y_true, y_pred, q):
    """
    Pinball loss for quantile q.
    y_true: actual value
    y_pred: predicted value (the quantile estimate)
    q: target quantile level (0 to 1)
    """
    residual = y_true - y_pred
    return np.where(residual >= 0, q * residual, (q - 1) * residual)

# Visualize loss curves for different quantile levels
u = np.linspace(-50, 50, 1000)  # Residuals (y - y_hat)
quantiles = [0.10, 0.25, 0.50, 0.75, 0.90]
colors = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4", "#9467bd"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for q, color in zip(quantiles, colors):
    loss = pinball_loss(y_true=0, y_pred=-u, q=q)  # y_true=0, vary y_pred
    ax.plot(u, loss, color=color, linewidth=2.5, label=f"q = {q:.2f}")
ax.axvline(0, color="black", linestyle="--", alpha=0.4, linewidth=1)
ax.set_xlabel("Residual u = y − ŷ (+ means you undershot)")
ax.set_ylabel("Pinball loss")
ax.set_title("Pinball Loss Curves\n(steeper right side → predicts higher quantile)")
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 25)

# Right: asymmetry for q=0.80
ax = axes[1]
q = 0.80
u_pos = u[u >= 0]
u_neg = u[u < 0]

ax.plot(u_pos, q * u_pos, color="#1f77b4", linewidth=3,
        label=f"Underprediction: slope = {q:.2f}")
ax.plot(u_neg, (q - 1) * u_neg, color="#d62728", linewidth=3,
        label=f"Overprediction: slope = {1-q:.2f}")
ax.axvline(0, color="black", linestyle="--", alpha=0.4)

# Annotate specific values
example_u = 20
ax.annotate(f"+20 units low\nloss = {q * example_u:.1f}",
            xy=(example_u, q * example_u),
            xytext=(30, q * example_u + 2),
            fontsize=9, color="#1f77b4",
            arrowprops=dict(arrowstyle="->", color="#1f77b4"))
ax.annotate(f"-20 units high\nloss = {(1-q) * example_u:.1f}",
            xy=(-example_u, (q - 1) * (-example_u)),
            xytext=(-48, (1-q) * example_u + 2),
            fontsize=9, color="#d62728",
            arrowprops=dict(arrowstyle="->", color="#d62728"))

ax.set_xlabel("Residual u = y − ŷ")
ax.set_ylabel("Pinball loss")
ax.set_title(f"q = {q:.2f}: Underpredicting is {q/(1-q):.0f}× more costly\n"
             f"→ Optimal predictor is the {q:.0%} quantile")
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 20)

plt.tight_layout()
plt.show()

In [ ]:
# Prove that the pinball loss minimizer is the quantile
# Generate a skewed distribution (more realistic for bakery data)
demand_dist = np.random.gamma(shape=4, scale=30, size=100_000) + 50

print("Distribution: Gamma(4, 30) + 50 (right-skewed, like bakery demand)")
print(f"Mean: {demand_dist.mean():.1f}, Median: {np.median(demand_dist):.1f}")
print()

# For each quantile level, find the minimizer of the expected pinball loss
# and compare to the empirical quantile
quantile_levels = [0.10, 0.25, 0.50, 0.75, 0.80, 0.90, 0.95]

print(f"{'Quantile':>10} {'Empirical':>12} {'Pinball min':>12} {'Match':>8}")
print("-" * 45)

for q in quantile_levels:
    # Empirical quantile
    empirical_q = np.percentile(demand_dist, q * 100)
    
    # Find the value that minimizes expected pinball loss
    candidates = np.linspace(demand_dist.min(), demand_dist.max(), 1000)
    expected_losses = [
        pinball_loss(demand_dist, c, q).mean()
        for c in candidates
    ]
    minimizer = candidates[np.argmin(expected_losses)]
    
    match = abs(empirical_q - minimizer) < 2  # within 2 units
    print(f"{q:>10.2f} {empirical_q:>12.1f} {minimizer:>12.1f} {'✓' if match else '✗':>8}")

print()
print("The minimizer of E[pinball_loss(q)] equals the q-th quantile.")
print("This is what MQLoss trains the neural network to produce.")

## Part 2: Effect of level= Settings

Different `level=` configurations give different amounts of uncertainty information. More levels = smoother uncertainty bands but longer training time.

In [ ]:
# Train three models with different level= settings
level_configs = {
    "Sparse: [80, 90]": [80, 90],
    "Medium: [50, 80, 90]": [50, 80, 90],
    "Dense: [50, 60, 70, 80, 90]": [50, 60, 70, 80, 90],
}

trained_models = {}

for config_name, levels in level_configs.items():
    model = NHITS(
        h=14,
        input_size=28,
        loss=MQLoss(level=levels),
        max_steps=300,
        random_seed=42,
    )
    nf = NeuralForecast(models=[model], freq="D")
    nf.fit(df=train_df)
    forecast = nf.predict()
    trained_models[config_name] = {"levels": levels, "forecast": forecast}
    print(f"Trained: {config_name} → columns: {[c for c in forecast.columns if 'NHITS' in c]}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, (config_name, data) in zip(axes, trained_models.items()):
    forecast = data["forecast"]
    levels = data["levels"]
    
    # Plot intervals from widest to narrowest
    sorted_levels = sorted(levels, reverse=True)
    base_alpha = 0.15
    alpha_step = 0.12
    
    for i, level in enumerate(sorted_levels):
        lo_col = f"NHITS-lo-{level}"
        hi_col = f"NHITS-hi-{level}"
        if lo_col in forecast.columns:
            ax.fill_between(
                forecast["ds"],
                forecast[lo_col],
                forecast[hi_col],
                alpha=base_alpha + i * alpha_step,
                color="steelblue",
                label=f"{level}%"
            )
    
    ax.plot(forecast["ds"], forecast["NHITS"],
            color="steelblue", linewidth=2, label="Mean")
    
    # Actual values
    ax.plot(test_df["ds"].values[:14], test_df["y"].values[:14],
            "k^", markersize=5, label="Actual", zorder=5)
    
    ax.set_title(config_name)
    ax.set_xlabel("Date")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(alpha=0.3)
    if ax == axes[0]:
        ax.set_ylabel("Baguettes sold")
    ax.legend(fontsize=8)

plt.suptitle("Effect of level= Settings on Fan Chart Smoothness", fontsize=12)
plt.tight_layout()
plt.show()

print("More levels → smoother gradient → more informative uncertainty visualization.")
print("All three models capture the same overall uncertainty range.")
print("The difference is resolution, not accuracy.")

## Part 3: Calibration — Does 80% Mean 80%?

A well-calibrated model should have empirical coverage close to the nominal level. This is the quantile forecast equivalent of a confidence interval actually containing the true value X% of the time.

In [ ]:
# Train a model for calibration evaluation using cross-validation
model_cal = NHITS(
    h=7,
    input_size=28,
    loss=MQLoss(level=[50, 80, 90]),
    max_steps=500,
    random_seed=42,
)

nf_cal = NeuralForecast(models=[model_cal], freq="D")

# Use cross-validation to get multiple test windows
# n_windows=4 gives 4 test periods of h=7 days each
cv_results = nf_cal.cross_validation(
    df=baguettes,
    n_windows=4,
    step_size=7,
)

print(f"Cross-validation results: {len(cv_results)} forecast rows")
print(cv_results[["ds", "y", "NHITS", "NHITS-lo-80", "NHITS-hi-80"]].head(10).to_string(index=False))

In [ ]:
# Compute empirical coverage for each nominal level
nominal_levels = [50, 80, 90]

coverage_results = []
for level in nominal_levels:
    lo_col = f"NHITS-lo-{level}"
    hi_col = f"NHITS-hi-{level}"
    
    in_interval = (
        (cv_results["y"] >= cv_results[lo_col]) &
        (cv_results["y"] <= cv_results[hi_col])
    )
    
    empirical_coverage = in_interval.mean()
    coverage_results.append({
        "nominal_level": level / 100,
        "empirical_coverage": empirical_coverage,
        "gap": empirical_coverage - level / 100,
    })

coverage_df = pd.DataFrame(coverage_results)
print("Calibration results:")
print(f"{'Nominal':>10} {'Empirical':>12} {'Gap':>8} {'Status':>12}")
print("-" * 45)
for _, row in coverage_df.iterrows():
    gap = row["gap"]
    if abs(gap) < 0.05:
        status = "Well-calibrated"
    elif gap > 0:
        status = "Overconfident?"
    else:
        status = "Conservative"
    print(f"{row['nominal_level']:>10.0%} {row['empirical_coverage']:>12.1%} "
          f"{gap:>+8.1%} {status:>12}")

In [ ]:
# Calibration plot: reliability diagram
# For a well-calibrated model, empirical coverage should fall on the diagonal

# Compute coverage at many quantile levels
quantile_levels_fine = np.arange(0.05, 1.0, 0.05)
empirical_coverages = []

for alpha in quantile_levels_fine:
    # Compute the alpha-quantile from the 80% interval endpoints
    # (approximate: use linear interpolation between lo-80 = 0.10 and hi-80 = 0.90)
    lo = cv_results["NHITS-lo-90"]   # 5th percentile
    hi = cv_results["NHITS-hi-90"]   # 95th percentile
    mean = cv_results["NHITS"]
    
    # Linearly interpolate to get the alpha-quantile
    if alpha <= 0.5:
        q_alpha = lo + (mean - lo) * (alpha / 0.05) / 10
    else:
        q_alpha = mean + (hi - mean) * ((alpha - 0.5) / 0.05) / 10
    
    coverage = (cv_results["y"] <= q_alpha).mean()
    empirical_coverages.append(coverage)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], "k--", linewidth=1.5, label="Perfect calibration")
ax.plot(quantile_levels_fine, empirical_coverages,
        "o-", color="steelblue", linewidth=2, markersize=5, label="NHITS + MQLoss")

# Mark the nominal levels
for level in [50, 80, 90]:
    row = coverage_df[coverage_df["nominal_level"] == level / 100].iloc[0]
    ax.plot(row["nominal_level"], row["empirical_coverage"],
            "r*", markersize=12, zorder=5)

ax.set_xlabel("Nominal quantile level")
ax.set_ylabel("Empirical coverage")
ax.set_title("Calibration Plot (Reliability Diagram)\n"
             "Points near the diagonal = well-calibrated")
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("Red stars mark the nominal levels (50%, 80%, 90%).")
print("Points above the diagonal: intervals are too wide (conservative).")
print("Points below the diagonal: intervals are too narrow (overconfident).")

## Part 4: MQLoss vs DistributionLoss

DistributionLoss assumes a parametric form for the distribution (Normal, Negative Binomial, etc.). For count data like bakery transactions, Negative Binomial is often more appropriate than assuming normality.

In [ ]:
# Visualize the empirical distribution of bakery demand
# to motivate the choice between Normal and NegativeBinomial
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

from scipy import stats

demand = train_df["y"].values

# Left: histogram with Normal fit
ax = axes[0]
ax.hist(demand, bins=40, alpha=0.6, color="steelblue", density=True,
        label="Actual demand")
mu, sigma = stats.norm.fit(demand)
x = np.linspace(demand.min(), demand.max(), 200)
ax.plot(x, stats.norm.pdf(x, mu, sigma), "r-", linewidth=2.5,
        label=f"Normal fit\nμ={mu:.0f}, σ={sigma:.0f}")
ax.set_title("Daily Baguette Demand vs Normal Distribution")
ax.set_xlabel("Baguettes sold")
ax.set_ylabel("Density")
ax.legend()
ax.grid(alpha=0.3)

# Right: Q-Q plot to check normality
ax = axes[1]
probplot_result = stats.probplot(demand, dist="norm")
theoretical_q = probplot_result[0][0]
sample_q = probplot_result[0][1]
ax.plot(theoretical_q, sample_q, "o", color="steelblue", alpha=0.6, markersize=4)
ax.plot(
    [theoretical_q.min(), theoretical_q.max()],
    [probplot_result[1][0] + probplot_result[1][1] * theoretical_q.min(),
     probplot_result[1][0] + probplot_result[1][1] * theoretical_q.max()],
    "r-", linewidth=2
)
ax.set_title("Normal Q-Q Plot\n(deviation from line = non-normal)")
ax.set_xlabel("Theoretical quantiles")
ax.set_ylabel("Sample quantiles")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Test for normality
stat, p_value = stats.shapiro(demand[:50])  # Shapiro-Wilk (sample)
print(f"Shapiro-Wilk normality test (p={p_value:.4f}):")
if p_value < 0.05:
    print("  p < 0.05 → significant evidence against normality")
    print("  NegativeBinomial or MQLoss (nonparametric) may be more appropriate")
else:
    print("  p > 0.05 → no strong evidence against normality")

# Check skewness
skew = stats.skew(demand)
print(f"\nSkewness: {skew:.3f}")
print("Values > 0.5 suggest right skew. Normal distribution may underestimate the right tail.")

In [ ]:
# Train models with different loss functions and compare calibration
loss_configs = {
    "MQLoss": MQLoss(level=[80, 90]),
    "Normal": DistributionLoss("Normal", level=[80, 90]),
    "NegativeBinomial": DistributionLoss("NegativeBinomial", level=[80, 90]),
}

comparison_results = {}

for loss_name, loss_fn in loss_configs.items():
    model = NHITS(
        h=7,
        input_size=28,
        loss=loss_fn,
        max_steps=400,
        random_seed=42,
    )
    nf = NeuralForecast(models=[model], freq="D")
    cv = nf.cross_validation(df=baguettes, n_windows=4, step_size=7)
    
    # Get column names regardless of model name prefix
    lo80_col = [c for c in cv.columns if "lo-80" in c][0]
    hi80_col = [c for c in cv.columns if "hi-80" in c][0]
    lo90_col = [c for c in cv.columns if "lo-90" in c][0]
    hi90_col = [c for c in cv.columns if "hi-90" in c][0]
    
    cov80 = (
        (cv["y"] >= cv[lo80_col]) & (cv["y"] <= cv[hi80_col])
    ).mean()
    cov90 = (
        (cv["y"] >= cv[lo90_col]) & (cv["y"] <= cv[hi90_col])
    ).mean()
    
    # Average interval width (80% interval)
    avg_width = (cv[hi80_col] - cv[lo80_col]).mean()
    
    comparison_results[loss_name] = {
        "coverage_80": cov80,
        "coverage_90": cov90,
        "avg_width_80": avg_width,
    }
    print(f"{loss_name}: 80% cov={cov80:.1%}, 90% cov={cov90:.1%}, avg width={avg_width:.0f}")

In [ ]:
# Summary table
print("\nComparison: MQLoss vs DistributionLoss")
print("=" * 65)
print(f"{'Loss Function':>20} {'80% Coverage':>15} {'90% Coverage':>15} {'Avg Width (80%)'!s:>16}")
print("-" * 65)

targets = {"coverage_80": 0.80, "coverage_90": 0.90}

for loss_name, res in comparison_results.items():
    gap80 = res["coverage_80"] - 0.80
    gap90 = res["coverage_90"] - 0.90
    print(f"{loss_name:>20} {res['coverage_80']:>14.1%} {res['coverage_90']:>14.1%} "
          f"{res['avg_width_80']:>15.0f}")

print(f"{'Target':>20} {'80.0%':>15} {'90.0%':>15}")
print()
print("Closer to the target coverage = better calibration.")
print("Narrower width = more informative (if calibration is maintained).")
print("NegativeBinomial is often best for integer count data.")

## Part 5: Quantile Crossing — A Warning

In theory, quantiles should be properly ordered: the 90th percentile should always be above the 80th percentile. In practice, neural networks can produce crossings, especially with sparse training data or short training runs.

In [ ]:
# Check for quantile crossings in our forecast
model_check = NHITS(
    h=14,
    input_size=28,
    loss=MQLoss(level=[50, 80, 90]),
    max_steps=500,
    random_seed=42,
)
nf_check = NeuralForecast(models=[model_check], freq="D")
nf_check.fit(df=train_df)
forecast_check = nf_check.predict()

# Check that all pairs of quantiles are properly ordered
pairs = [
    ("lo-90", "lo-80"),   # 5th pct should be below 10th pct
    ("lo-80", "lo-50"),   # 10th pct should be below 25th pct
    ("lo-50", "NHITS"),   # 25th pct should be below mean
    ("NHITS", "hi-50"),   # mean should be below 75th pct
    ("hi-50", "hi-80"),   # 75th pct should be below 90th pct
    ("hi-80", "hi-90"),   # 90th pct should be below 95th pct
]

print("Quantile ordering checks (no crossings expected):")
all_ok = True
for lo_suffix, hi_suffix in pairs:
    lo_col = f"NHITS-{lo_suffix}" if lo_suffix != "NHITS" else "NHITS"
    hi_col = f"NHITS-{hi_suffix}" if hi_suffix != "NHITS" else "NHITS"
    
    crossings = (forecast_check[lo_col] > forecast_check[hi_col]).sum()
    status = "OK" if crossings == 0 else f"WARNING: {crossings} crossings"
    print(f"  {lo_suffix:>8} < {hi_suffix:<8}: {status}")
    if crossings > 0:
        all_ok = False

print()
if all_ok:
    print("All quantiles properly ordered. No crossings detected.")
else:
    print("Crossings detected. Solutions:")
    print("  1. Train for more steps")
    print("  2. Use DistributionLoss (quantiles derived from CDF — can't cross)")
    print("  3. Post-process with isotonic regression")

In [ ]:
section_divider("Summary")

## Summary

In this notebook we explored MQLoss in depth:

1. **Pinball loss** is an asymmetric loss where the cost of underprediction vs overprediction is controlled by the quantile level $q$. The minimizer of the expected pinball loss is the $q$-th quantile — verified empirically on bakery data.

2. **level= settings:** More levels produce smoother fan charts. The choice of levels is a trade-off between visualization quality and training compute. `level=[80, 90]` is the practical default.

3. **Calibration:** Empirical coverage should match nominal coverage. Use cross-validation to check. Deviations indicate overconfidence (intervals too narrow) or conservatism (intervals too wide).

4. **MQLoss vs DistributionLoss:** MQLoss is nonparametric and generally safe. DistributionLoss is more appropriate when the data matches a known distribution family. For count data like bakery demand, NegativeBinomial is often better calibrated than Normal.

5. **Quantile crossing:** Check that quantiles are properly ordered. Crossings can occur with limited training; use DistributionLoss or post-processing to fix.

**Key limitation remains:** All of these produce marginal quantiles. They are correct per-day but insufficient for multi-day aggregations. That is the subject of Module 3.

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])